# 03 — Baseline Classifier: CNN-LSTM

Trains a supervised CNN-LSTM classifier on PADS RightWrist Relaxed task (HC vs PD).  
Uses 5-fold stratified cross-validation at subject level.  
This model replaces the autoencoder as the base model for activation extraction in notebook 04.

**Input:** preprocessed windows `(N, 6, 200)` + labels + subject IDs  
**Output:** trained model checkpoints per fold + fold metrics JSON

## 0. Setup

In [ ]:
import sys
import os

# Add repo root to path so src/ imports work
sys.path.append(os.path.abspath('..'))

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src.data.dataset import PADSDataset
from src.data.folds import load_fold_splits, get_fold_windows
from src.models.cnn_lstm import CNNLSTM
from src.training.trainer import train_model
from src.training.metrics import compute_metrics, aggregate_fold_metrics, print_fold_results, print_summary
from src.utils.io import save_metrics, load_checkpoint

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. Load preprocessed data

These files are produced by `notebooks/01_data_processing.ipynb`.

In [ ]:
DATA_DIR  = '../data/processed'
FOLDS_PATH = os.path.join(DATA_DIR, 'fold_splits.pkl')

windows     = np.load(os.path.join(DATA_DIR, 'windows.npy'))      # (N, 6, 200)
labels      = np.load(os.path.join(DATA_DIR, 'labels.npy'))       # (N,) 0=HC 1=PD
subject_ids = np.load(os.path.join(DATA_DIR, 'subject_ids.npy'))  # (N,)

print(f"Windows:     {windows.shape}")
print(f"Labels:      {labels.shape}  — HC: {(labels==0).sum()}  PD: {(labels==1).sum()}")
print(f"Subjects:    {len(np.unique(subject_ids))}")

folds = load_fold_splits(FOLDS_PATH)
print(f"Folds loaded: {len(folds)}")

## 2. Hyperparameters

In [ ]:
# Model
N_CHANNELS   = 6
N_TIMESTEPS  = 200
N_CLASSES    = 2
CNN_FILTERS  = (64, 128)
KERNEL_SIZE  = 3
LSTM_HIDDEN  = 128
LSTM_LAYERS  = 2
DROPOUT      = 0.3

# Training
BATCH_SIZE   = 32
LR           = 1e-3
MAX_EPOCHS   = 100
PATIENCE     = 15
WEIGHT_DECAY = 1e-4

# Class imbalance — weight the loss to account for HC/PD ratio
# ~790 HC windows vs ~2760 PD windows → weight HC more
N_HC = (labels == 0).sum()
N_PD = (labels == 1).sum()
class_weights = torch.tensor(
    [N_PD / N_HC, 1.0], dtype=torch.float32
).to(device)
print(f"Class weights — HC: {class_weights[0]:.2f}  PD: {class_weights[1]:.2f}")

## 3. Training — 5-fold cross-validation

In [ ]:
fold_metrics = []

for fold_idx, fold in enumerate(folds):
    print(f"\n{'='*50}")
    print(f"FOLD {fold_idx + 1} / {len(folds)}")
    print(f"{'='*50}")

    # --- Get fold windows ---
    (train_win, train_lab,
     val_win,   val_lab,
     test_win,  test_lab) = get_fold_windows(
        windows, labels, subject_ids, fold
    )

    # For classifier training, use ALL train windows (HC + PD)
    # get_fold_windows returns HC-only for train/val by default
    # We need to override for supervised training
    train_mask = np.isin(subject_ids, fold['train_subjects'])
    val_mask   = np.isin(subject_ids, fold['val_subjects'])

    train_win = windows[train_mask]
    train_lab = labels[train_mask]
    val_win   = windows[val_mask]
    val_lab   = labels[val_mask]

    print(f"Train: {train_win.shape[0]} windows "
          f"(HC: {(train_lab==0).sum()}, PD: {(train_lab==1).sum()})")
    print(f"Val:   {val_win.shape[0]} windows "
          f"(HC: {(val_lab==0).sum()}, PD: {(val_lab==1).sum()})")
    print(f"Test:  {test_win.shape[0]} windows "
          f"(HC: {(test_lab==0).sum()}, PD: {(test_lab==1).sum()})")

    # --- Normalize using train stats ---
    from src.data.preprocessing import compute_normalization_stats, apply_normalization
    mean, std = compute_normalization_stats(train_win)
    train_win = apply_normalization(train_win, mean, std)
    val_win   = apply_normalization(val_win,   mean, std)
    test_win  = apply_normalization(test_win,  mean, std)

    # --- DataLoaders ---
    train_ds = PADSDataset(train_win, train_lab)
    val_ds   = PADSDataset(val_win,   val_lab)
    test_ds  = PADSDataset(test_win,  test_lab)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              shuffle=True,  drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                              shuffle=False)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE,
                              shuffle=False)

    # --- Model ---
    model = CNNLSTM(
        n_channels=N_CHANNELS,
        n_timesteps=N_TIMESTEPS,
        n_classes=N_CLASSES,
        cnn_filters=CNN_FILTERS,
        kernel_size=KERNEL_SIZE,
        lstm_hidden=LSTM_HIDDEN,
        lstm_layers=LSTM_LAYERS,
        dropout=DROPOUT,
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
    )
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    # --- Train ---
    model, history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        criterion=criterion,
        fold_idx=fold_idx,
        model_name='cnn_lstm',
        device=device,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        checkpoint_dir='../models/checkpoints',
        verbose=True,
    )

    # --- Evaluate on test set ---
    model.eval()
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)
            logits = model(x)
            probs = torch.softmax(logits, dim=1)[:, 1]  # PD probability
            all_probs.append(probs.cpu().numpy())
            all_labels.append(y.numpy())

    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)

    metrics = compute_metrics(all_labels, all_probs)
    fold_metrics.append(metrics)
    print_fold_results(fold_idx, metrics)

print(f"\n{'='*50}")
agg = aggregate_fold_metrics(fold_metrics)
print_summary(agg)

## 4. Save results

In [ ]:
save_metrics(fold_metrics, 'cnn_lstm_cv_results.json',
             results_dir='../results/metrics')
print("Done.")

## 5. Quick sanity check — model architecture and parameter count

In [ ]:
model_check = CNNLSTM(
    n_channels=N_CHANNELS, n_timesteps=N_TIMESTEPS, n_classes=N_CLASSES,
    cnn_filters=CNN_FILTERS, kernel_size=KERNEL_SIZE,
    lstm_hidden=LSTM_HIDDEN, lstm_layers=LSTM_LAYERS, dropout=DROPOUT
)
total_params = sum(p.numel() for p in model_check.parameters())
print(model_check)
print(f"\nTotal parameters: {total_params:,}")